## 1.Loading Needed Libraries

In [1]:
import glob
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 8)

## 2.Loading Files

In [2]:
files = glob.glob('../data/raw/item_properties_part?.csv')

In [3]:
events = pd.read_csv('../data/raw/events.csv')
category_tree = pd.read_csv('../data/raw/category_tree.csv')
item_props = pd.concat([pd.read_csv(f)  for f in files], ignore_index=True)

## 3.Processing Events

Dropping duplicates, transaction id column, bot/abnormal users and less interacted items.

In [4]:
events = events.drop_duplicates()
events = events.drop('transactionid', axis=1)

In [5]:
weighted_mapping = {
    'view': 1,
    'addtocart': 4,
    'transaction': 9
}

events['weight'] = events['event'].map(weighted_mapping)

## 4.Filling Category Tree

Filling missing parentid rows with -1.

In [6]:
category_tree['parentid'] = category_tree['parentid'].fillna(-1)

#### Sanity Check

In [7]:
print(f'Number of missing data:\n{category_tree.isna().sum().to_string()}')

Number of missing data:
categoryid    0
parentid      0


## 5.Cleaning Item Properties Part 1 and 2

Getting rid of rows with numerical values, keeping only 'available' and 'categoryid'.

In [8]:
item_props = item_props[item_props['property'].isin(['categoryid', 'available'])]

#### Sanity Check

In [9]:
print(f'Frequency of each property:\n')
print(item_props.value_counts('property').to_string())

Frequency of each property:

property
available     1503639
categoryid     788214


## 6.Creating Train/Test Set

In [10]:
events = events.sort_values('timestamp')

In [11]:
cutoff = events['timestamp'].quantile(0.8)

train = events[events['timestamp'] < cutoff]
test = events[events['timestamp'] >= cutoff]

In [12]:
user_counts = train['visitorid'].value_counts()
item_counts = train['itemid'].value_counts()

train['user_interaction_count'] = train['visitorid'].map(user_counts)
train['item_interaction_count'] = train['itemid'].map(item_counts)

train = train[(train['user_interaction_count'] >= 2) & (train['user_interaction_count'] <= 20)]
train = train.drop('user_interaction_count', axis=1)

train = train[train['item_interaction_count'] >= 2]
train = train.drop('item_interaction_count', axis=1)

test = test[test['visitorid'].isin(train['visitorid']) & test['itemid'].isin(train['itemid'])]

# Keep only new user-item pairs (exclude re-interactions already in train)
train_pairs = train[['visitorid', 'itemid']].drop_duplicates()
test = test.merge(train_pairs, on=['visitorid', 'itemid'], how='left', indicator=True)
test = test[test['_merge'] == 'left_only'].drop(columns='_merge')

In [13]:
user_enc = LabelEncoder()
item_enc = LabelEncoder()

train['user_idx'] = user_enc.fit_transform(train['visitorid'])
test['user_idx'] = user_enc.transform(test['visitorid'])


train['item_idx'] = item_enc.fit_transform(train['itemid'])
test['item_idx'] = item_enc.transform(test['itemid'])

train = train.drop(['visitorid', 'itemid'], axis=1)
test = test.drop(['visitorid', 'itemid'], axis=1)

#### Sanity Check

In [14]:
train_shape = train.shape
test_shape = test.shape

print(f'# of rows: {train_shape[0]}')
print(f'# of columns: {train_shape[1]}')
print('_'*50 + '\n')
print(f'# of rows: {test_shape[0]}')
print(f'# of columns: {test_shape[1]}')

# of rows: 1093131
# of columns: 5
__________________________________________________

# of rows: 20731
# of columns: 5


## 7.Saving Processed Datas

In [15]:
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

events.to_csv('../data/processed/events_processed.csv', index=False)
category_tree.to_csv('../data/processed/category_tree_processed.csv', index=False)
item_props.to_csv('../data/processed/item_props_processed.csv', index=False)

## 8.Saving Encoders

In [16]:
with open('../models/user_encoder.pkl', 'wb') as f:
    pickle.dump(user_enc, f)
with open('../models/item_encoder.pkl', 'wb') as f:
    pickle.dump(item_enc, f)